# Text Classification with the Bag of Words

**Notebook 1 of 3.** You train a model that reads a poem and names its author, Emily Dickinson or Walt Whitman.

A classifier needs numbers, so every poem becomes a vector: first word counts (the **bag of words**), then **TF-IDF** weights, which favor rare, distinctive words over ubiquitous ones. A logistic regression learns which words point to which poet, and a grid search finds the best way of building those vectors. The same approach sorts emails into spam and news into reliable and fake.

## Setup

In [ ]:
# standard library
import os
import re
import warnings
from pathlib import Path

# data stack
import numpy as np
import pandas as pd

# visualization stack
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# nlp stack
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# ml stack
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

In [ ]:
# NLTK's data: WordNet, the dictionary the lemmatizer looks words up in, and the English
# stop word list (11 MB together, downloaded once). It goes in the project rather than the
# home directory: NLTK refuses to read a folder that other accounts can write to, and a
# default umask often leaves ~/nltk_data so. The environment variable passes the location
# to the parallel processes the grid search of section 5 starts.
NLTK_DATA = Path("nltk_data").resolve()
NLTK_DATA.mkdir(mode=0o700, exist_ok=True)
os.environ["NLTK_DATA"] = str(NLTK_DATA)
nltk.data.path = [str(NLTK_DATA)]
nltk.download("wordnet", download_dir=str(NLTK_DATA), quiet=True)
nltk.download("stopwords", download_dir=str(NLTK_DATA), quiet=True)

# Two warnings that would otherwise repeat on every one of section 5's hundreds of fits.
# NLTK reports a stray ~/nltk_data that other accounts can write to, which this notebook
# does not read; scikit-learn reports that a stemmed document is matched against an
# unstemmed stop word list, which section 5 measures on purpose.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="nltk")
warnings.filterwarnings("ignore", message="Your stop_words may be inconsistent")

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)


## 1. Load the Poems and Split Them

`data/poems.csv` holds two Project Gutenberg books, *Poems by Emily Dickinson* and Whitman's *Leaves of Grass*. One row is one document: a poem, or a numbered section of one of Whitman's long ones. The model reads `text` and predicts `author`.

Two sets, 80 % and 20 %. Every choice here is made on the training set by **cross-validation**, so there is no separate validation set and the test set is scored once, at the end.

In [ ]:
poems = pd.read_csv(Path("data") / "poems.csv")
print(poems["author"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    poems["text"], poems["author"], test_size=0.2, stratify=poems["author"], random_state=42
)
print(f"\nTraining: {len(X_train)}, test: {len(X_test)} documents")

Whitman wrote 59 % of the documents, so a model that always answers *Whitman* is right 59 % of the time. Every model of this notebook has to beat this share.

## 2. The Bag of Words by Hand

**The bag of words**
>- every word of the **vocabulary**, the words of all training documents, becomes a column;
>- every document becomes a row holding the number of times it contains each word;
>- word order is discarded, as if each document's words were shaken in a bag, hence the name.

| Term | Meaning |
|---|---|
| document | one text, the unit that gets classified; here one poem |
| corpus | the collection of documents, here all the poems |
| token | one unit of text, usually a word; **tokenizing** splits a text into tokens |
| vocabulary | the set of different tokens in the corpus |
| stop words | frequent words that carry little meaning, such as `the` and `of` |
| n-gram | a sequence of n neighboring tokens: `sea` is a unigram, `the sea` a bigram |
| vectorizing | turning a text into a vector of numbers |

Build one for three famous lines first: lowercase, tokenize, collect the vocabulary, count.

In [ ]:
lines = pd.Series(
    [
        "Success is counted sweetest by those who ne'er succeed.",
        "I celebrate myself, and sing myself.",
        "I sing the body electric.",
    ],
    index=["Dickinson", "Whitman 1", "Whitman 2"],
)
lowercase = lines.str.lower()
lowercase

### Tokenize with Regular Expressions

`str.split()` splits at spaces only, so `myself,` keeps its comma and counts as a different word from `myself`. A **regular expression** is a small language for describing a pattern in text: you write the shape the text has, and [`re.findall`](https://docs.python.org/3/library/re.html#re.findall) returns every piece that matches.

One pattern carries this notebook, `\b\w\w+\b`, the one scikit-learn uses: `\w` is a word character, `+` means one or more of them and `\b` is a word boundary, so together they match a word of two or more characters. Write patterns as raw strings, `r"..."`, so Python passes the backslashes to `re` unchanged.

In [ ]:
print(re.findall(r"\b\w\w+\b", lowercase["Whitman 1"]))

It drops one-letter words such as `i` and `a`, and the apostrophe in `ne'er` splits it into `ne` and `er`.

Both are choices rather than accidents, and they are affordable because this notebook has one narrow job: telling two poets apart. A pattern is never *the* way to split text, only one way, and the decision comes back with every task — notebook 02 keeps letters only, dropping the digits this pattern keeps, and says why where it does it.

Tokenize all three lines with the pattern and collect the vocabulary:

In [ ]:
# Tokenize every line: one list of tokens per line
tokens = {}
for name, text in lowercase.items():
    tokens[name] = re.findall(r"\b\w\w+\b", text)
    print(f"{name}: {tokens[name]}")

# The vocabulary: every different token of the three lines, sorted alphabetically
vocabulary = set()
for line_tokens in tokens.values():
    vocabulary.update(line_tokens)
vocabulary = sorted(vocabulary)
print(len(vocabulary), "words in the vocabulary:", vocabulary)

In [ ]:
# Count every word of the vocabulary in every line: one row per line, one column per word
rows = []
for line_tokens in tokens.values():
    row = {}
    for word in vocabulary:
        row[word] = line_tokens.count(word)  # 0 if the word does not occur in the line
    rows.append(row)

bag_by_hand = pd.DataFrame(rows, index=list(tokens))
bag_by_hand

Many text pipelines remove the **stop words** at this point. There is no canonical list: NLTK ships 198 entries, which hold the everyday prepositions and conjunctions but leave the more literary ones out — `upon`, `within`, `among`, `although`, `yet` — along with the modals `could`, `would`, `may` and `must`. scikit-learn's own list of 318 has them all. This notebook takes both and runs them through the tokenizer's pattern, so an entry like `you're` becomes `you` and `re`. Whether removing them helps is a question for the grid search in section 5.

In [ ]:
# Both lists, deduplicated, then tokenized with the same pattern as the documents
both_lists = set(stopwords.words("english")) | set(ENGLISH_STOP_WORDS)
stop_words = sorted({word for entry in both_lists for word in re.findall(r"\b\w\w+\b", entry)})
print(len(stop_words), "stop words, for example:", stop_words[:12])

# Which words of the three lines are stop words?
[word for word in vocabulary if word in stop_words]

### Stemming and Lemmatization

Every word form is a column of its own, so `sing`, `sings` and `singing` are three unrelated features. Two classic steps fold them together: **stemming** chops the ending off by rule, fast and not always leaving a real word, and **lemmatization** looks the word up in a dictionary, so the token stays readable.

| Word | Stem | Lemma |
|---|---|---|
| `sings` | `sing` | `sings` |
| `leaves` | `leav` | `leaf` |
| `heavenly` | `heavenli` | `heavenly` |
| `bees` | `bee` | `bee` |

They disagree both ways: the stemmer cuts `leaves` to a non-word, the lemmatizer leaves `sings` alone because it folds nouns unless told the word is a verb. Section 5 lets the search decide.

In [ ]:
# PorterStemmer is pure Python; WordNetLemmatizer needs the WordNet data, downloaded once
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

for word in ["sings", "singing", "leaves", "heavenly", "bees", "waters", "dying"]:
    print(f"{word:10s} stem: {stemmer.stem(word):10s} lemma: {lemmatizer.lemmatize(word)}")

### A Word Cloud for Each Poet

A **word cloud** prints every word at a size set by its count, so with the stop words gone it shows at a glance what a text is about. Count the words of each poet first.

In [ ]:
def content_words(author):
    """Count the words of one poet's poems, leaving the stop words out."""
    text = " ".join(poems.loc[poems["author"] == author, "text"])
    tokens = re.findall(r"\b\w\w+\b", text.lower())
    return pd.Series([token for token in tokens if token not in stop_words]).value_counts()


frequencies = {author: content_words(author) for author in ["Dickinson", "Whitman"]}
for author, counts in frequencies.items():
    print(f"{author}: {counts.sum():,} content words, most frequent {list(counts.head(6).index)}")

Those counts are the bag of words of each poet. `generate_from_frequencies` takes them as they are, so nothing is counted twice. Draw the two side by side:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (author, counts) in zip(axes, frequencies.items()):
    cloud = WordCloud(width=800, height=400, background_color="white", random_state=42)
    cloud.generate_from_frequencies(counts)
    ax.imshow(cloud, interpolation="bilinear")
    ax.set_title(author)
    ax.axis("off")
plt.show()

**Result.** Dickinson's most frequent content words are `like`, `little`, `day`, `know` and `away`; Whitman's are `love`, `old`, `life`, `shall` and `thee`. Whitman is the bulkier poet: 57,691 content words against Dickinson's 12,609.

What the clouds mostly show is how much the two share: `day` and `life` stand near the top of both, and neither poet names a subject the other avoids. If the content words barely separate them, something else has to — which is what section 6 finds. Note too that 344 stop words do not catch everything: `like` and `shall` survive, since a general-purpose list is not built for nineteenth-century verse.

### The Same with `CountVectorizer`

[`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) does every step above — lowercase, tokenize with `\b\w\w+\b`, learn the vocabulary, count. Like a scaler, `fit` learns the vocabulary from the training texts and `transform` counts any text. The parameters used here:

| Parameter | Default | What it does |
|---|---|---|
| `lowercase` | `True` | Lowercases before tokenizing, so `Sea` and `sea` are one word. |
| `stop_words` | `None` | A list removes those words; `"english"` removes scikit-learn's own 318-word list. |
| `tokenizer` | `None` | A function that splits a document into tokens, in place of the pattern. |
| `ngram_range` | `(1, 1)` | The shortest and longest n-grams: `(1, 2)` adds bigrams to the single words. |

In [ ]:
counter = CountVectorizer()
counts = counter.fit_transform(lines)
bag_sklearn = pd.DataFrame(counts.toarray(), columns=counter.get_feature_names_out(), index=lines.index)
print("Same as by hand:", bag_sklearn.equals(bag_by_hand))
bag_sklearn

`fit_transform` returns a **sparse matrix**: it stores only the non-zero entries and their positions, and `toarray()` turns it into a normal array for display. With whole books the vocabulary has thousands of words, and almost every entry of the matrix is zero:

In [ ]:
counts_train = CountVectorizer().fit_transform(X_train)
n_documents, n_words = counts_train.shape
print(f"{n_documents} documents x {n_words:,} words")
print(f"Non-zero entries: {counts_train.nnz:,} of {n_documents * n_words:,} ({counts_train.nnz / (n_documents * n_words):.2%})")

**Result.** The 865 training documents hold 12,729 different words, so the matrix has 11 million entries — but only 71,213 of them, 0.65 %, are not zero, because a poem uses a few dozen to a few hundred of those words. Storing only these saves over 99 % of the memory, which is why scikit-learn returns sparse matrices for text.

## 3. TF-IDF: Weigh the Words

Word counts treat every word alike, but `the` occurs in almost every poem of both poets while a word occurring in a few says more when it appears. **TF-IDF**, term frequency times inverse document frequency, weighs them accordingly. For a word $t$ in document $d$, with $n$ training documents of which $\text{df}(t)$ contain $t$:

$$\text{tf-idf}(t, d) = \text{tf}(t, d) \cdot \text{idf}(t), \qquad \text{idf}(t) = \ln\frac{1 + n}{1 + \text{df}(t)} + 1$$

The term frequency is the count, as in the bag of words. The inverse document frequency is largest for a word in one document and smallest, 1, for one in every document, with the $+ 1$ keeping a ubiquitous word from vanishing. Each row is then divided by its **L2 norm**, so a long and a short document with the same mix get the same vector.

**Your turn:** compute the TF-IDF vector of `I sing the body electric.` by hand, for the three lines of section 2.

In [ ]:
n = len(lines)
# Document frequencies of the four words of "I sing the body electric." ("I" is not a token)
df_words = (bag_by_hand[["body", "electric", "sing", "the"]] > 0).sum()

# TODO: the idf of each word, with the formula above (use np.log)
idf = ...
# TODO: every word occurs once in the line, so tf-idf = idf; divide by the L2 norm of the vector
by_hand = ...

tfidf_lines = TfidfVectorizer().fit(lines)
sklearn_row = tfidf_lines.transform(["I sing the body electric."]).toarray()[0]
sklearn_weights = pd.Series(sklearn_row, index=tfidf_lines.get_feature_names_out())[df_words.index]
pd.DataFrame({"df": df_words, "idf": idf, "by hand": by_hand, "scikit-learn": sklearn_weights}).round(3)

**Result.** `body`, `electric` and `the` occur in one of the three lines and get idf $\ln(4/2) + 1 = 1.693$; `sing` occurs in two and gets the smaller $\ln(4/3) + 1 = 1.288$. Dividing by the vector length $\sqrt{3 \cdot 1.693^2 + 1.288^2} = 3.20$ gives 0.529 and 0.402, scikit-learn's values. `sing` weighs least because it is the least distinctive: it appears in another line too.

[`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) is a `CountVectorizer` followed by these weights, so it takes every parameter above and adds four of its own:

| Parameter | Default | What it does |
|---|---|---|
| `use_idf` | `True` | Multiplies by the idf; `False` keeps the plain term frequencies, still normalized. |
| `sublinear_tf` | `False` | `True` replaces the count by $1 + \ln(\text{tf})$, so a word repeated many times weighs less. |
| `smooth_idf` | `True` | The $1 +$ in the fraction of the formula above. |
| `norm` | `"l2"` | The row scaling, here to length 1. |

Section 5 searches the first two.

## 4. A Pipeline, Scored by Cross-Validation

A [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) chains the vectorizer and the classifier, so `fit` learns both from the training texts alone. It takes `(name, step)` pairs, and section 5 addresses its settings by those names. [Logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) suits text: thousands of sparse features, one readable coefficient per word. `solver="liblinear"` matches the default accuracy many times faster, which matters once section 5 fits it hundreds of times.

**Cross-validation** replaces a single validation set: [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) splits the training poems into 5 folds and scores each in turn, so the mean rests on every poem rather than one arbitrary fifth.

In [ ]:
tfidf_pipeline = Pipeline(
    [("vectorizer", TfidfVectorizer()), ("classifier", LogisticRegression(solver="liblinear"))]
)
scores = cross_val_score(tfidf_pipeline, X_train, y_train, cv=5)
print("The five folds:", scores.round(3))
print(f"TF-IDF, default settings: {scores.mean():.3f} +- {scores.std():.3f}")

count_pipeline = Pipeline(
    [("vectorizer", CountVectorizer()), ("classifier", LogisticRegression(solver="liblinear"))]
)
print(f"Plain word counts:        {cross_val_score(count_pipeline, X_train, y_train, cv=5).mean():.3f}")

**Result.** TF-IDF with its default settings reaches 0.904, **below** the plain word counts at 0.918. TF-IDF is not automatically better: its rows have length 1, so its values are far smaller than counts, and the logistic regression needs weaker regularization — a larger `C` — to make the same use of them. That is one of the choices the next section searches instead of guessing.

## 5. Search the Choices with a Grid

Every step from text to numbers was a choice, and each one is a hypothesis worth testing:

- **remove the stop words**, or keep them;
- **weigh by idf** (`use_idf`), or use the term frequencies alone;
- **dampen repetition** (`sublinear_tf`), or count every occurrence in full;
- **fold the word forms together** by stemming or lemmatizing, or leave them apart;
- and `C`, how hard the logistic regression is regularized.

One at a time would miss the combinations: section 4 showed TF-IDF looking worse than counts until `C` moves with it. [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) fits every combination with 5-fold cross-validation.

The three normalizations become three `tokenizer` functions. A tokenizer replaces the splitting step only, so `stop_words` still applies; a function passed as `analyzer` would switch it off silently.

In [ ]:
def words(document):
    """Split a document into lowercase tokens of two or more word characters."""
    return re.findall(r"\b\w\w+\b", document.lower())


def stems(document):
    """Tokenize, then cut every token down to its stem."""
    return [stemmer.stem(token) for token in words(document)]


def lemmas(document):
    """Tokenize, then replace every token by its dictionary form."""
    return [lemmatizer.lemmatize(token) for token in words(document)]

Each entry of the grid names a step of the pipeline, two underscores and a parameter, so `vectorizer__use_idf` is the `use_idf` of the step named `vectorizer`. The five lists make 72 combinations, 360 fits with five folds each, which takes about a minute.

In [ ]:
parameter_grid = {
    "vectorizer__tokenizer": [words, stems, lemmas],
    "vectorizer__stop_words": [None, stop_words],
    "vectorizer__use_idf": [True, False],
    "vectorizer__sublinear_tf": [False, True],
    "classifier__C": [1, 10, 100],
}

# token_pattern=None tells the vectorizer that the tokenizer replaces its own pattern.
# n_jobs=-1 spreads the fits over every core.
search = GridSearchCV(
    Pipeline(
        [
            ("vectorizer", TfidfVectorizer(token_pattern=None)),
            ("classifier", LogisticRegression(solver="liblinear")),
        ]
    ),
    parameter_grid,
    cv=5,
    n_jobs=-1,
)
search.fit(X_train, y_train)

print(f"{len(search.cv_results_['params'])} combinations fitted")
print(f"Best cross-validated accuracy: {search.best_score_:.3f}")

`cv_results_` holds every combination with its mean accuracy over the five folds.

In [ ]:
results = pd.DataFrame(search.cv_results_)

# The two settings that are objects rather than numbers, as readable words
results["tokenizer"] = results["param_vectorizer__tokenizer"].map(lambda f: f.__name__)
results["stop words"] = results["param_vectorizer__stop_words"].map(
    lambda value: "removed" if value is not None else "kept"
)
results.shape

`cv_results_` carries a column for every parameter and several timings. Keep the five settings and the mean accuracy, and sort:

In [ ]:
columns = {
    "tokenizer": "tokenizer",
    "stop words": "stop words",
    "param_vectorizer__use_idf": "use_idf",
    "param_vectorizer__sublinear_tf": "sublinear_tf",
    "param_classifier__C": "C",
    "mean_test_score": "accuracy",
}
table = results[list(columns)].rename(columns=columns)

# Number the rows by position, so the index reads 1, 2, 3 for first, second, third
best_eight = table.sort_values("accuracy", ascending=False).head(8).round(3)
best_eight.index = range(1, len(best_eight) + 1)
best_eight.index.name = "rank"

print(f"The weakest of the {len(table)} combinations scored {table['accuracy'].min():.3f}")
best_eight

**Result.** The best of the 72 combinations reaches 0.972 against 0.904 for section 4's defaults: stemmed tokens, stop words **kept**, idf on, `sublinear_tf=True`, `C=10`. The ranking says more than its first row:

- every one of the top eight weighs by idf and dampens repetition with `sublinear_tf`, while the weakest of the 72 scores 0.846;
- `C=1`, section 4's default, appears nowhere in the top eight. That, not TF-IDF itself, was section 4's handicap;
- ranks 1 to 6 all **keep** the stop words. The best combination that removes them reaches 0.954;
- the three tokenizers are hard to separate: stemming takes the top row at 0.972, lemmatizing 0.971 and plain words 0.965.

Two surprise: **normalizing the word forms bought almost nothing**, though it is standard advice, and **removing the stop words made the model worse**. Section 6 explains the second.

## 6. Which Words Decide?

The search refitted its best combination on the whole training set and kept it as `best_estimator_`. That is the model to ask, since it is the one the next section will be judged on.

A logistic regression adds up one coefficient per word, weighted by the TF-IDF value of that word. The classes are sorted alphabetically, so `Dickinson` is class 0 and `Whitman` class 1: a **positive** coefficient pushes a poem towards Whitman, a **negative** one towards Dickinson.

In [ ]:
best_model = search.best_estimator_
vectorizer = best_model.named_steps["vectorizer"]
classifier = best_model.named_steps["classifier"]

# One coefficient per feature, sorted: Dickinson's at the bottom, Whitman's at the top
coefficients = pd.Series(classifier.coef_[0], index=vectorizer.get_feature_names_out()).sort_values()
top = pd.concat([coefficients.head(15), coefficients.tail(15)])
top.tail(5).round(2)

The 15 strongest of each, as a bar chart:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top.index, top.values, color=["tab:blue"] * 15 + ["tab:orange"] * 15)
ax.set(
    xlabel="Coefficient (negative: Dickinson, positive: Whitman)",
    title="The 15 strongest words of the best model, for each poet",
)
plt.show()

**Result.** The deciding words are not the poets' themes but short **function words**. Whitman is recognized by `and`, `of`, `or`, `you`, `me`, `them` — his long cumulative lines join lists with *and*, and he addresses *you*. Dickinson by `when`, `then`, `so`, `if`, `could`, `would`, the words of short conditional statements, with `bee` her one theme in the top 15.

**This is why removing the stop words cost accuracy:** 22 of these 30 are on the list. Which way it goes depends on what separates the classes. Where that is **topic** — sports against politics — the content words decide and dropping the stop words loses nothing; where it is **style**, as here, the stop words *are* the evidence.

`wa` and `arm` are stems, not words — and `wa` shows why stemming and a stop word list sit awkwardly together: `was` is on the list, its stem is not, so it survives the removal meant to take it out.

## 7. The Test Set

The test set has taken part in no decision: the grid search saw the training poems only. Score the best model on it once and show the errors in a [confusion matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html).

In [ ]:
print(f"Test accuracy: {search.score(X_test, y_test):.3f}")
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, cmap="Blues", colorbar=False)
plt.grid(False)
plt.show()

**Result.** The test accuracy is 0.972, the same as the cross-validated accuracy, so there is no sign that searching 72 combinations overfitted the training poems. Six of the 217 test documents are misclassified: 3 poems by Dickinson predicted as Whitman, and 3 by Whitman as Dickinson.

## Summary

- A regular expression describes the shape of a piece of text: `re.findall(r"\b\w\w+\b", text)` returns every word of two or more characters.
- A bag of words turns each document into a row of counts, over 99 % of them zero. TF-IDF then multiplies each count by $\ln\frac{1 + n}{1 + \text{df}} + 1$ and scales each row to length 1, so distinctive words weigh more than ubiquitous ones.
- Cross-validation scores on every training document in turn, so no separate validation set is needed. A grid search tests the choices in combination rather than one at a time, because they interact: 0.904 to 0.972 here.
- The best model's coefficients name the deciding words — short function words, which is why removing the stop words costs accuracy here though it often helps when topic decides the class.

Next, continue with [02 - Word Embeddings](02_word_embeddings.ipynb).

## Sources

- The poems are Project Gutenberg eBooks [#12242](https://www.gutenberg.org/ebooks/12242) and [#1322](https://www.gutenberg.org/ebooks/1322), public domain in the USA; `data/build_poems.py` shows how `poems.csv` was made from them.
- The bag of words built by hand here follows the Bag of Words notebook of the Classical NLP lesson.
- TF-IDF with a logistic regression on top is the baseline that *Hands-On Large Language Models* (Alammar and Grootendorst, O'Reilly, 2024) recommends measuring language models against, in chapter 4. No chapter of that book builds it, so this notebook does.

The repository [README](README.md) lists every source and licence.